# Mimic a real dataset, then prove the copy is faithful

You have a real table you cannot share. PII, contracts, governance. You want a
synthetic copy your team can pass around freely, train on, and demo with.

Two questions decide whether that copy is worth anything:

1. Does it match the real data, not just column by column, but in the relationships
   between columns?
2. Can you prove it, with a number, instead of eyeballing a histogram?

Misata answers both. `mimic` learns each column's distribution and the correlation
structure across columns, then generates a fresh dataset that reuses none of the
real rows. `fidelity_report` scores how close that copy is to the original.

We use scikit-learn's diabetes dataset here because it is real, bundled offline, and
recognisable. Swap in your own CSV with one line.

In [1]:
# pip install "misata>=0.8.1.4" scikit-learn
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
from sklearn.datasets import load_diabetes
import misata

real = load_diabetes(as_frame=True).frame   # 442 patients, 10 features + disease-progression target
real.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


## 1. Mimic it in one line

`mimic` profiles every column, detects the correlations between them, and generates
a synthetic twin. No model to configure, no real values copied.

In [2]:
syn = misata.mimic(real, rows=len(real), seed=7)['table']
syn.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.032580,-0.044642,0.031251,-0.033213,0.049499,0.039887,-0.043401,0.071210,0.082258,-0.001078,247
1,0.035464,0.050680,0.000616,0.053147,0.046923,-0.012927,0.061001,-0.039493,0.070985,0.031516,109
2,0.080482,-0.044642,0.059485,0.043131,-0.048653,-0.044889,-0.032356,-0.002592,-0.020850,0.040343,103
3,-0.091884,-0.044642,0.015536,-0.026328,0.000674,-0.008710,-0.034582,0.021241,0.003848,-0.008582,128
4,-0.007272,0.050680,-0.089709,-0.040099,-0.091939,-0.100173,-0.011753,-0.058467,-0.049494,-0.043818,59


## 2. Score the copy against the real data

`fidelity_report` returns one overall number plus a breakdown: per-column shape
(marginals), the correlation structure (joint), and ML efficacy. ML efficacy trains
a model on the synthetic data and tests it on the real data, then compares against a
real-trained baseline. A ratio near 1.0 means the synthetic data carries the same
predictive signal as the real data.

In [3]:
report = misata.fidelity_report(syn, real, target_column='target')
print(report.summary())

Overall fidelity: 0.964  (1.000 = identical to real)

  Marginals (per-column shape) : 0.930
  Correlations (joint structure): 0.963
  ML efficacy (TSTR/TRTR)      : 1.000  (synthetic-trained 0.342 vs real-trained 0.338, regression on 'target')

  Lowest-fidelity columns:
    s4                       0.771  (ks)
    s6                       0.880  (ks)
    bp                       0.925  (ks)
    s1                       0.943  (ks)
    target                   0.948  (ks)


## 3. Why this needs joint structure, not just marginals

This is the part most synthetic data tools miss. It is easy to match each column on
its own. It is hard to keep the relationships between them. Here is the same dataset
mimicked two ways: marginals only, versus marginals plus correlations.

In [4]:
import numpy as np
from misata.profiler import DataProfiler
from misata.simulator import DataSimulator

# Marginal-only: profile, then strip the learned correlations.
schema = DataProfiler().profile(real, table_name='table')
schema.tables[0].correlations = []
sim = DataSimulator(schema); sim.rng = np.random.default_rng(7)
syn_marginal = {n: d for n, d in sim.generate_all()}['table']

rep_marginal = misata.fidelity_report(syn_marginal, real, target_column='target')
rep_joint    = report  # from the cell above (correlation-aware, the default)

print(f"{'':22}{'marginals only':>16}{'+ correlations':>16}")
print(f"{'overall fidelity':22}{rep_marginal.overall_score:>16.3f}{rep_joint.overall_score:>16.3f}")
print(f"{'correlation match':22}{rep_marginal.correlation_score:>16.3f}{rep_joint.correlation_score:>16.3f}")
print(f"{'ML efficacy (TSTR)':22}{rep_marginal.ml_efficacy['efficacy_ratio']:>16.3f}{rep_joint.ml_efficacy['efficacy_ratio']:>16.3f}")

                        marginals only  + correlations
overall fidelity                 0.588           0.964
correlation match                0.834           0.963
ML efficacy (TSTR)               0.000           1.000


Read the ML efficacy row. With marginals only, a model trained on the synthetic data
is useless on the real data: the columns look right but the relationships that carry
the signal are gone. Once correlations are learned, a model trained purely on the
synthetic copy predicts real disease progression as well as one trained on real data.

That is the difference between data that looks real and data that is useful.

## 4. The privacy win

A faithful copy is only safe if it does not smuggle real records through. Check how
many synthetic rows are exact matches of real rows.

In [5]:
leaks = real.merge(syn, how='inner')
print(f'Exact real rows leaked into the synthetic copy: {len(leaks)} of {len(syn)}')

Exact real rows leaked into the synthetic copy: 0 of 442


Zero. Every synthetic row is freshly generated. You can share this dataset, commit
it to a repo, or hand it to a vendor without exposing a single real patient.

## What this does, and does not, capture

Honest scope, because fidelity should never be a black box:

- **Captured well:** per-column distributions, value ranges, null rates, category
  frequencies, and linear correlations between numeric columns.
- **Not yet captured:** nonlinear and geometric structure. Mimic a dataset built on
  latitude and longitude clusters, for example, and `fidelity_report` will show a
  lower ML efficacy. That is the point of the score: it tells you when to trust the
  copy and when not to, instead of leaving you guessing.

### Bring your own data

```python
real = pd.read_csv('your_table.csv')
syn  = misata.mimic(real, rows=len(real))['table']
print(misata.fidelity_report(syn, real, target_column='your_label').summary())
```

Misata is open source: `pip install misata` and [github.com/rasinmuhammed/misata](https://github.com/rasinmuhammed/misata)